# Reflective and Catadioptric Systems

Reflective optics — mirrors — are chromatic-aberration-free and can cover large
apertures efficiently. The NSQ engine supports spherical and conic mirrors via
`MirrorConfig`, which feeds into `scene.add_mirror()`.

This notebook covers:
1. Spherical vs. parabolic mirror spot comparison
2. Solar concentrator — parabolic mirror + irradiance analysis
3. Two-mirror Cassegrain-like relay
4. Catadioptric system — combining a mirror and a refractive lens
5. Far-field radiation pattern of a reflective source

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from optiland.coordinate_system import CoordinateSystem
from optiland.nonsequential import (
    NSQScene, Spectrum,
    CollimatedSourceConfig, PointSourceConfig,
    IrradianceDetectorConfig, FarFieldDetectorConfig,
    LensConfig, MirrorConfig, SurfaceConfig,
    SpecularBRDF,
)

spec = Spectrum.monochromatic(0.55)

## 1. Mirror Orientation Convention

By default a mirror's reflecting surface faces `+z`. To make it face an
incoming beam from `−z`, rotate it 180° around x:
```python
CoordinateSystem(z=0, rx=np.pi)   # concave face toward −z
```
The focal length of a conic mirror is `f = |radius| / 2`.

## 2. Spherical vs. Parabolic Mirror

A **spherical mirror** (`conic=0`) suffers from spherical aberration for
on-axis collimated input: marginal rays focus closer to the mirror than
paraxial rays. A **parabolic mirror** (`conic=-1`) eliminates this aberration
and focuses a collimated on-axis beam to a perfect geometrical point.

In [2]:
def mirror_spot(conic, aperture_radius=25, n_rays=40_000):
    # Focal length = 100 mm (R = -200 mm, concave toward -z)
    scene_m = NSQScene()
    scene_m.add_source(
        'S', CoordinateSystem(z=-100),
        CollimatedSourceConfig(spec, total_flux=1.0, aperture_radius=aperture_radius),
    )
    scene_m.add_mirror(
        'M', CoordinateSystem(z=0, rx=np.pi),
        MirrorConfig(radius=-200, conic=conic, aperture_radius=aperture_radius + 2),
    )
    scene_m.add_detector(
        'D', CoordinateSystem(z=-100),
        IrradianceDetectorConfig(width=8, height=8, num_pixels_x=128, num_pixels_y=128),
    )
    r = scene_m.trace(num_rays=n_rays, seed=42)
    return r.detectors['D']

irr_sphere  = mirror_spot(conic=0.0)
irr_parab   = mirror_spot(conic=-1.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, irr, title in zip(axes, [irr_sphere, irr_parab],
                          ['Spherical (conic=0)', 'Parabolic (conic=-1)']):
    im = ax.imshow(irr.irradiance, origin='lower', cmap='hot', aspect='equal',
                   extent=[irr.x_coords[0], irr.x_coords[-1],
                            irr.y_coords[0], irr.y_coords[-1]])
    plt.colorbar(im, ax=ax, label='W/mm²')
    ax.set_title(f'{title}\nPeak: {irr.irradiance.max():.3f} W/mm²')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
plt.suptitle('Focal spot: spherical vs. parabolic mirror', fontsize=12)
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\AppData\Local\Temp\ipykernel_18160\3043387605.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Solar Concentrator

A parabolic dish concentrator captures parallel solar radiation and focuses it
onto a small receiver. The concentration ratio is the detector area divided by
the mirror area.

In [3]:
# Solar irradiance ≈ 1000 W/m² = 0.1 W/cm² = 0.001 W/mm²
# Mirror aperture: 50 mm radius → 7854 mm² → flux = 7.85 W
mirror_radius = 50.0
solar_flux    = 0.001 * np.pi * mirror_radius**2   # W

# Broadband solar-like spectrum (visible + NIR)
spec_solar = Spectrum(
    wavelengths=np.array([0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]),
    weights=np.array([0.6, 1.0, 1.1, 1.0, 0.9, 0.7, 0.5]),
)

scene_solar = NSQScene()
scene_solar.add_source(
    'Sun', CoordinateSystem(z=-200),
    CollimatedSourceConfig(spec_solar, total_flux=solar_flux,
                           aperture_radius=mirror_radius),
)
scene_solar.add_mirror(
    'Dish', CoordinateSystem(z=0, rx=np.pi),
    MirrorConfig(radius=-400, conic=-1.0, aperture_radius=mirror_radius + 2),
)
# Receiver at focus (f = R/2 = 200 mm)
scene_solar.add_detector(
    'Receiver', CoordinateSystem(z=-200),
    IrradianceDetectorConfig(width=10, height=10, num_pixels_x=128, num_pixels_y=128),
)

result_solar = scene_solar.trace(num_rays=80_000, seed=42)
irr_solar = result_solar.detectors['Receiver']

concentration = irr_solar.irradiance.max() / (solar_flux / (np.pi * mirror_radius**2))
print(f"Mirror flux in      : {solar_flux:.2f} W")
print(f"Receiver flux       : {irr_solar.total_flux:.4f} W")
print(f"Peak concentration  : {concentration:.0f}× (vs. ambient 0.001 W/mm²)")

fig = irr_solar.plot(cmap='hot')
plt.title(f'Solar concentrator — peak {irr_solar.irradiance.max():.3f} W/mm²')
plt.tight_layout()
plt.show()
plt.close(fig)

Mirror flux in      : 7.85 W
Receiver flux       : 0.0269 W
Peak concentration  : 18× (vs. ambient 0.001 W/mm²)


C:\Users\kdani\AppData\Local\Temp\ipykernel_18160\975368594.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Catadioptric System — Mirror + Lens

A catadioptric system combines a primary reflective element with a refractive
corrector. Here we use a concave mirror as the primary and a singlet lens as a
field-correcting relay.

In [4]:
scene_cat = NSQScene()
scene_cat.add_source(
    'S', CoordinateSystem(z=-150),
    CollimatedSourceConfig(spec, total_flux=1.0, aperture_radius=20.0),
)
# Primary mirror: f = 75 mm
scene_cat.add_mirror(
    'PM', CoordinateSystem(z=0, rx=np.pi),
    MirrorConfig(radius=-150, conic=-1.0, aperture_radius=22.0),
)
# Field corrector lens near the focus
scene_cat.add_lens(
    'FC', CoordinateSystem(z=-50),
    LensConfig(r1=-80, r2=80, thickness=4, material='N-BK7', front_aperture_radius=15.0),
)
# Detector
scene_cat.add_detector(
    'D', CoordinateSystem(z=-90),
    IrradianceDetectorConfig(width=8, height=8, num_pixels_x=128, num_pixels_y=128),
)

result_cat = scene_cat.trace(num_rays=50_000, seed=42)
irr_cat = result_cat.detectors['D']

fig = irr_cat.plot(cmap='hot')
plt.title(f'Catadioptric system | {irr_cat.num_rays_hit:,} rays on detector')
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\backends\array_backend.py:300: RuntimeWarning: invalid value encountered in multiply
  dx = det_t_min * rays.L
C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\backends\array_backend.py:301: RuntimeWarning: invalid value encountered in multiply
  dy = det_t_min * rays.M


C:\Users\kdani\AppData\Local\Temp\ipykernel_18160\269616946.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Far-Field Radiation Pattern

A `FarFieldDetector` captures the **angular distribution** of light exiting the
system. Useful for characterising the beam divergence from a reflective source.

In [5]:
# LED-like point source reflected and collimated by a parabolic mirror
scene_farfield = NSQScene()
scene_farfield.add_source(
    'LED', CoordinateSystem(z=-100),   # at focus of mirror
    PointSourceConfig(spec, total_flux=1.0, half_angle_deg=90),  # hemisphere
)
scene_farfield.add_mirror(
    'PM', CoordinateSystem(z=0, rx=np.pi),
    MirrorConfig(radius=-200, conic=-1.0, aperture_radius=30.0),
)
# Far-field detector: large aperture captures collimated output beam
scene_farfield.add_detector(
    'FF', CoordinateSystem(z=-150),
    FarFieldDetectorConfig(num_theta=60, num_phi=180),
)

result_ff = scene_farfield.trace(num_rays=80_000, seed=42)
ff = result_ff.detectors['FF']

# Radial (azimuthally averaged) far-field profile
radial = ff.intensity.sum(axis=1)
half_max = radial.max() / 2.0
# Approximate FWHM
above_half = ff.theta[radial >= half_max]
fwhm = above_half[-1] - above_half[0] if len(above_half) > 1 else 0.0

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ff.theta, radial / radial.max())
ax.axhline(0.5, color='r', linestyle='--', label=f'Half-max (FWHM ≈ {fwhm:.1f}°)')
ax.set_xlabel('θ [deg]')
ax.set_ylabel('Normalised intensity')
ax.set_title('Far-field pattern: LED + parabolic mirror collimator')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
plt.close(fig)

print(f"Approximate FWHM beam divergence: {fwhm:.1f}°")

Approximate FWHM beam divergence: 31.5°


C:\Users\kdani\AppData\Local\Temp\ipykernel_18160\3915950673.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

- `MirrorConfig(radius, conic, aperture_radius)` — conic constant: 0=sphere, -1=paraboloid
- Flip mirror to face incoming beam: `CoordinateSystem(z=..., rx=np.pi)`
- Focal length of a mirror: `f = |radius| / 2`
- Parabolic mirrors eliminate on-axis spherical aberration for collimated input
- Mix `add_mirror` and `add_lens` in the same scene for catadioptric systems
- `FarFieldDetector` measures beam divergence (FWHM) and angular distribution